### Getting the deputies data

We will work with the [LVI period of the Deputies Chamber ("*Cámara de Diputadas y Diputados*")](https://www.bcn.cl/historiapolitica/corporaciones/periodo_detalle?inicio=2022-03-11&fin=2026-03-10&periodo=1990-2026&cam=Diputados#:~:text=Parlamentarios), the dates of this period are from 2022-03-11 to 2026-03-10.

There's a difference between Legislative Period ("*Periodo legislativo*") (4 year term of the Chamber), and the Legislation ("*Legislatura*") (1 year subdivision in the periodo Legislativo), we will work on the Legislative Period.

We want to get the deputy ID (dipid), names, gender, date of birth, actual political afiliation and past political affiliations.

This Legislative Period ends 2026-03-10, next day a new cohort of deputies will start, that's why I will use a call to a specific Period, and a call to the actual deputies. Because that will break after 03.11.


From the database that the National Congress Library ("*Biblioteca Nacional del Congreso*") we can see that the Legislative Period ("*Periodo legislativo*") we want to work with is the LVI (56th). But in their inner database the period is numbered with ID=10

[This is where we can find the endpoints from the Deputies' Chamber](https://www.camara.cl/transparencia/datosAbiertos.aspx)

In [ ]:
# Import chunk
import requests
import sqlite3
import pandas as pd
import xml.etree.ElementTree as et
from datetime import datetime


In [10]:
# Clean function to handle None values and strip whitespace, this process will be used for all the text fields we extract from the XML
def clean_text(text):
    """Clean text by stripping whitespace (and handling none values)"""
    if text is None:
        return None
    return text.strip()

In [24]:
# this is the endpoint URL that we will work on
url = "https://opendata.camara.cl/camaradiputados/WServices/WSDiputado.asmx/retornarDiputadosXPeriodo"

# calling the API
# the id we are usign now is 10, that's the Period 2022-2026
response = requests.get(url, params={"prmPeriodoID":10}, timeout=10)

# answer is a xml
deputies_xml = response.text
#deputies_xml

In [25]:
# parse XML using ElementTree (setting the data in a tree structure)
deputies_parsed = et.fromstring(deputies_xml)
deputies_parsed

# since in the tree structure the namespace gets inhered for every children and grandchildren of the root, we have to clean and reduce it
# While working with tags we need to find a way to work also with the namespaces
ns = {"ns": "http://opendata.camara.cl/camaradiputados/v1"}

[Here is a description on how the data is stored in the Deputies' Chamber (in Spanish)](https://opendata.camara.cl/camaradiputados/v1/Index_DiputadosPeriodoColeccion1.html)
I decided not to get the District information from here because they use a different codification to what's used publically in Chile, and for all the Districts, Regions and Communes in Chile that wasn't feasible to hardcode

In [27]:
# We create list to store the information
deputies_list = []
parties_list = []
militancies_list = []


# loop over each deputy
for dp in deputies_parsed.findall("ns:DiputadoPeriodo", ns):
    # first the information on the term boundaries
    term_start = dp.findtext("ns:FechaInicio", default=None, namespaces=ns)
    term_end = dp.findtext("ns:FechaTermino", default=None, namespaces=ns)
    # now we know that the term_start is really the end of the electoral process

    # now nested inside there's the concrete information on the deputy
    diputado = dp.find("ns:Diputado", ns)
    if diputado is None:
        continue

    diputado_id = diputado.findtext("ns:Id", default=None, namespaces=ns)
    if diputado_id is None:
        continue
    dipid = int(diputado_id)

    # the sex element has text and attribute
    sex_element = diputado.find("ns:Sexo", ns)
    sex_text = sex_element.text if sex_element is not None else None # the XXX is not None else None or Continue (as seen in Party) is here to have extra security for future executions. But for now it can also run without it
    sex_value = sex_element.attrib.get("Valor") if sex_element is not None else None
    
    deputies_list.append({
        "dipid": dipid,
        "nombre": clean_text(diputado.findtext("ns:Nombre", default=None, namespaces=ns)), # some deputies have two names, but they are all in the same tag
        "apellido_paterno": clean_text(diputado.findtext("ns:ApellidoPaterno", default=None, namespaces=ns)),
        "apellido_materno": clean_text(diputado.findtext("ns:ApellidoMaterno", default=None, namespaces=ns)),
        "fecha_nacimiento": clean_text(diputado.findtext("ns:FechaNacimiento", default=None, namespaces=ns)),
        "sex_text": sex_text,
        "sex_value": sex_value
    })

    # now we proceeed with the information on the historical militancies
    militancias = diputado.find("ns:Militancias", ns)
    if militancias is None:
        continue
    for i in militancias.findall("ns:Militancia", ns): # we loop over each militancy they have
        i_start = i.findtext("ns:FechaInicio", default=None, namespaces=ns)
        i_end = i.findtext("ns:FechaTermino", default=None, namespaces=ns)

        partido = i.find("ns:Partido", ns)
        if partido is None:
            continue

        party_id = partido.findtext("ns:Id", default=None, namespaces=ns)
        party_name = partido.findtext("ns:Nombre", default=None, namespaces=ns)
        party_alias = partido.findtext("ns:Alias", default=None, namespaces=ns)

        # we save the information on the parties that had elected representatives in the LVI legislature
        parties_list.append({
            "party_id": party_id,
            "party_name": party_name,
            "party_alias": party_alias
        })

        # finally we save a table linking deputies to their parties
        militancies_list.append({
            "dipid": dipid,
            "party_id": party_id,
            "date_start": i_start,
            "date_end": i_end
        })
    
    # store the results in dataframes
    dip_df = pd.DataFrame(deputies_list)
    parties_df = pd.DataFrame(parties_list).drop_duplicates(subset=["party_id"])
    militancies_df = pd.DataFrame(militancies_list).drop_duplicates(subset=["dipid", "party_id", "date_start"])

In [ ]:
#dip_df

The reason that I searched for historical militancies was because I wanted to study changes in patterns when representatives changes parties, but I drop this idea for this iteration. This information was either way important in the analysis and dashboard part of this project.

## Getting District and Commune ("*Comuna*") Information from the BCN API

BCN API is the Api from the National Congress Library (BCN from the Spanish name). We call it to get the respective [administrative division](https://en.wikipedia.org/wiki/Administrative_divisions_of_Chile) of the deputies, which still works until 21.03.2026, but it is not reliable because some of the "active deputies" it shows are not Deputies anymore This supplements the existing data with geographic details.

[This the secondary API with the district, comuna and region information](https://datos.bcn.cl/catalogo/servicio/ServiciosWebHistoriaDeLaLey/ObtenerParlamentariosActivos)

In [12]:
def strip_namespaces(root):
    """
    Remove namespaces in XML so the tags can be accessed with simpler names like instead of 'ns:Diputado'.
    """
    for elem in root.iter():
        if "}" in elem.tag:
            elem.tag = elem.tag.split("}", 1)[1]
    return root

In [15]:
# BCN API endpoint for active parliamentary members
bcn_url = "https://datos.bcn.cl/catalogo/servicio/ServiciosWebHistoriaDeLaLey/ObtenerParlamentariosActivos"

try: # try to call the API because it is not always responsive
    # Call the BCN API
    bcn_response = requests.get(bcn_url, timeout=30) # timeout because the BCN API can be slow to respond
    bcn_response.raise_for_status() # check if the request was successful

    # here we parse the XML string response into a tree structure and strip the namespaces
    bcn_root = strip_namespaces(et.fromstring(bcn_response.text)) # we strip the namespaces to make it easier to work with the tags
    
    print("BCN API response received")
# in case of any error
except requests.RequestException as e:
    print(f"Error: {e}")
    bcn_root = None

BCN API response received


In [ ]:
#bcn_root.findall(".//Parlamentario")

In [20]:
# Parse the BCN XML response
districts_list = []


if bcn_root is not None: # parse only if the request and the parsing where successful
    
    # The response contains many Senators and Deputies, we only keep deputies
    for parl in bcn_root.findall(".//Parlamentario"):

        chamber = clean_text(parl.findtext(".//Camara/Descripcion"))
        if chamber != "Cámara de Diputados":
            continue

        # This id is the one that matches the dipid we have already
        dipid_text = clean_text(parl.findtext(".//IdEnCamaraDeOrigen"))
        if dipid_text is None:
            continue

        try:
            dipid = int(dipid_text)
        except ValueError:
            continue

        # Geographic information
        geo = parl.find(".//RepresentacionGeografica")

        district_name = None
        district_number = None   # kept for schema compatibility; not cleanly available here
        region = None
        comunas = [] # there are several comunas for each district

        # Geographic information is nested under RepresentacionGeografica.
        if geo is not None:
            division = geo.find(".//DivisionPoliticoAdministrativa")
            if division is not None:
                # BCN sometimes uses lowercase 'descripcion', sometimes 'Descripcion'
                district_name = (clean_text(division.findtext("descripcion"))
                                 or clean_text(division.findtext("Descripcion")))

            region = clean_text(geo.findtext(".//Region/Descripcion"))

            for comuna in geo.findall(".//Comunas/Comuna"):
                comuna_name = clean_text(comuna.findtext("Descripcion"))
                if comuna_name:
                    comunas.append(comuna_name)

        districts_list.append({
            "dipid": dipid,
            "district_name": district_name,
            "comunas": ", ".join(comunas) if comunas else None,
            "region": region
        })

# Build the districts DataFrame.
districts_df = pd.DataFrame(districts_list)

In [21]:
#districts_df

,dipid,district_name,comunas,region
0,1165,Distrito N° 8,"Cerrillos, Colina, Estación Central, Lampa, Ma...",Región Metropolitana De Santiago
1,1179,Distrito N° 8,"Cerrillos, Colina, Estación Central, Lampa, Ma...",Región Metropolitana De Santiago
2,1160,Distrito N° 10,"La Granja, Macul, Ñuñoa, Providencia, San Joaq...",Región Metropolitana De Santiago
3,1013,Distrito N° 26,"Ancud, Calbuco, Castro, Chaitén, Chonchi, Coch...",Región De Los Lagos
4,1076,Distrito N° 17,"Constitución, Curepto, Curicó, Empedrado, Hual...",Región Del Maule
...,...,...,...,...
150,1158,Distrito N° 5,"Andacollo, Canela, Combarbalá, Coquimbo, Illap...",Región De Coquimbo
151,1125,Distrito N° 8,"Cerrillos, Colina, Estación Central, Lampa, Ma...",Región Metropolitana De Santiago
152,991,Distrito N° 1,"Arica, Camarones, General Lagos, Putre",Región De Arica Y Parinacota
153,1059,Distrito N° 12,"La Florida, La Pintana, Pirque, Puente Alto, S...",Región Metropolitana De Santiago


In [36]:
print(len(dip_df))
print(len(districts_df))

157
155


There's a difference in the amount of rows we get from the BCN and the Deputies' Chamber. Those two extra in the BCN are [Tomás de Rementería](https://en.wikipedia.org/wiki/Tom%C3%A1s_de_Rementer%C3%ADa_Venegas), who started as deputy in this Legislative Period, but took a Senate position when another [Socialist Senator was removed from office]((https://en.wikipedia.org/wiki/Tom%C3%A1s_de_Rementer%C3%ADa_Venegas#:~:text=On%2016%20April%202025%2C%20the%20Political%20Committee%20of%20the%20Socialist%20Party%20ratified%20him%20to%20fill%20the%20seat%20vacated%20by%20Senator%20Isabel%20Allende%2C%20who%20had%20been%20removed%20from%20office%20by%20the%20Constitutional%20Court%2E%5B4%5D)). The other person is [Mercedes Bulnes](https://en.wikipedia.org/wiki/Mercedes_Bulnes), who [died in November 2024](https://en.wikipedia.org/wiki/Mercedes_Bulnes#Death) (while still in office)

## Creating SQL tables

We create the SQL tables and then upload the corresponding information

In [37]:
# before uploeading the information we translate the column names
deputies_sql = dip_df.rename(columns={
    "nombre": "name",
    "apellido_paterno": "lastname_1",
    "apellido_materno": "lastname_2",
    "fecha_nacimiento": "birth_date"
})

In [42]:
con = sqlite3.connect("../../data/chilean_polispeech.db")
con.execute("PRAGMA foreign_keys = ON;")  # Enable foreign key support
cur = con.cursor()

# Executescript allows to execute several SQL statements at once
con.executescript("""
    CREATE TABLE IF NOT EXISTS deputies (
        dipid INT PRIMARY KEY,
        name TEXT,
        lastname_1 TEXT,
        lastname_2 TEXT,
        birth_date TEXT,
        sex_value TEXT,
        sex_text TEXT
    );

    CREATE TABLE IF NOT EXISTS parties (
        party_id INT PRIMARY KEY,
        party_name TEXT,
        party_alias TEXT
    );

    CREATE TABLE IF NOT EXISTS militancies (
        dipid INT,
        party_id INT,
        date_start TEXT,
        date_end TEXT,
        PRIMARY KEY (dipid, party_id, date_start),
        FOREIGN KEY (dipid) REFERENCES deputies (dipid),
        FOREIGN KEY (party_id) REFERENCES parties (party_id)
    );

    CREATE TABLE IF NOT EXISTS districts (
        dipid INT PRIMARY KEY,
        district_number TEXT,
        district_name TEXT,
        comunas TEXT,
        region TEXT,
        data_source TEXT,
        FOREIGN KEY (dipid) REFERENCES deputies (dipid)
    );
    """)

con.commit()

# insert the data
deputies_sql.to_sql("deputies", con, if_exists="replace", index=False)
parties_df.to_sql("parties", con, if_exists="replace", index=False)
militancies_df.to_sql("militancies", con, if_exists="replace", index=False)
districts_df.to_sql("districts", con, if_exists="replace", index=False)
 


155

In [ ]:
#districts_df 

,dipid,district_name,comunas,region
0,1165,Distrito N° 8,"Cerrillos, Colina, Estación Central, Lampa, Ma...",Región Metropolitana De Santiago
1,1179,Distrito N° 8,"Cerrillos, Colina, Estación Central, Lampa, Ma...",Región Metropolitana De Santiago
2,1160,Distrito N° 10,"La Granja, Macul, Ñuñoa, Providencia, San Joaq...",Región Metropolitana De Santiago
3,1013,Distrito N° 26,"Ancud, Calbuco, Castro, Chaitén, Chonchi, Coch...",Región De Los Lagos
4,1076,Distrito N° 17,"Constitución, Curepto, Curicó, Empedrado, Hual...",Región Del Maule
...,...,...,...,...
150,1158,Distrito N° 5,"Andacollo, Canela, Combarbalá, Coquimbo, Illap...",Región De Coquimbo
151,1125,Distrito N° 8,"Cerrillos, Colina, Estación Central, Lampa, Ma...",Región Metropolitana De Santiago
152,991,Distrito N° 1,"Arica, Camarones, General Lagos, Putre",Región De Arica Y Parinacota
153,1059,Distrito N° 12,"La Florida, La Pintana, Pirque, Puente Alto, S...",Región Metropolitana De Santiago


### Exploratory chunks

In [6]:
# save the data so we don't have to download it again (in case we get blocked by the server)
# with open("../../data/diputados_periodo_10.xml", "w", encoding="utf-8") as f:
#     f.write(deputies_xml)

# Save the response for reference
    # with open("../../data/bcn_parlamentarios_activos.xml", "w", encoding="utf-8") as f:
    #     f.write(bcn_xml)